# 14. Графовая сеть (Graph WaveNet) — проверка пространственной гипотезы (Kaggle, GPU)

Одна и та же сеть с разными графами — честная проверка H2 и H3: что дают соседи, если модель умеет
использовать их на каждом временном слое, а не только через средние отклонения (как LightGBM).

| модель | граф |
|---|---|
| `gnn_none` | без графа — станции независимы (глубокая модель без пространства) |
| `gnn_line`, `gnn_dist`, `gnn_od_in` | один фиксированный граф |
| `gnn_od_dyn` | динамический граф OD: срез по времени суток и типу дня |
| `gnn_adapt` | граф выучивается из данных |
| `gnn_all` | линии + расстояние + OD + выучиваемый |

Все — с поправкой смещения, прогноз на все точки val и test.

**Настройки:** Accelerator — **GPU T4 x1**, Internet — On; Input — датасет `vkr-transit-data`.
Время ~3–4 ч (каждая модель ~25–35 мин). Уже посчитанные модели при повторном запуске пропускаются.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Проверка на утечку (несколько секунд)

In [ ]:
!cd {REPO} && python src/train_gnn.py --leak-test --graphs line dist od_in od_dyn --adaptive

## 2. Обучение и прогнозы

In [ ]:
RUNS = [
    ("gnn_none",   ""),
    ("gnn_line",   "--graphs line"),
    ("gnn_adapt",  "--adaptive"),
    ("gnn_all",    "--graphs line dist od_in --adaptive"),
    ("gnn_dist",   "--graphs dist"),
    ("gnn_od_in",  "--graphs od_in"),
    ("gnn_od_dyn", "--graphs od_dyn"),
]
for name, flags in RUNS:
    if os.path.exists(f"{R}/preds/{name}__test.npz"):
        print(f"{name}: уже посчитан"); continue
    t = time.time()
    !cd {REPO} && python src/train_gnn.py --name {name} {flags} --bias-correct
    print(f"=== {name}: {(time.time() - t) / 60:.0f} мин\n", flush=True)

## 3. Быстрая проверка качества

In [ ]:
!cd {REPO} && python src/baselines.py
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
m = pd.read_csv(f"{R}/metrics_val.csv")
C = m[m.subset == "все точки"]
H_ = ["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]
hz = C[C.slice.isin(H_)].pivot_table(index="model", columns="slice", values="wape")[H_]
display(hz.round(4).sort_values("горизонт h1"))
if "gnn_none" in hz.index:
    print("изменение к gnn_none (та же сеть без графа), %"); display(((hz / hz.loc["gnn_none"] - 1) * 100).round(1))

## 4. Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))